# Fake News Detection — Phase 3: Multi-Model Comparison

**Goal:** Compare 4 classifiers — Logistic Regression, Multinomial Naive Bayes, Linear SVM,
and Random Forest — on accuracy, precision, recall, F1-score, and cross-validation, then
select the best one for hyperparameter tuning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

sns.set_style('whitegrid')

## 1. Load Cleaned Data

In [ ]:
df = pd.read_csv('fake_news_data_clean.csv')
print('Shape:', df.shape)
df.head(2)

In [ ]:
df['label'].value_counts()

## 2. Vectorize & Split

In [ ]:
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=5000, min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print('Train shape:', X_train_tfidf.shape)
print('Test shape:', X_test_tfidf.shape)

## 3. Define Models

We train 4 classifiers with sensible defaults:

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes': MultinomialNB(),
    'Linear SVM': LinearSVC(max_iter=10000, random_state=42, dual='auto'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

## 4. Train, Predict & Collect Metrics

In [ ]:
results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    predictions[name] = y_pred

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True, target_names=['Factual', 'Fake'])

    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision (Fake)': round(report['Fake']['precision'], 4),
        'Recall (Fake)': round(report['Fake']['recall'], 4),
        'F1-score (Fake)': round(report['Fake']['f1-score'], 4),
        'F1-score (Factual)': round(report['Factual']['f1-score'], 4)
    })

    print(f'{name:25} Accuracy: {acc:.2%}  |  F1 Fake: {report["Fake"]["f1-score"]:.4f}  |  F1 Factual: {report["Factual"]["f1-score"]:.4f}')

results_df = pd.DataFrame(results)
results_df

## 5. Classification Reports (Detailed)

In [ ]:
for name in models:
    print(f'=' * 50)
    print(f'  {name}')
    print(f'=' * 50)
    print(classification_report(y_test, predictions[name], target_names=['Factual News', 'Fake News']))
    print()

## 6. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Factual', 'Fake'],
                yticklabels=['Factual', 'Fake'], ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()

## 7. Cross-Validation Comparison

5-fold CV on the full dataset for a more robust estimate.

In [ ]:
X_full_tfidf = tfidf.fit_transform(X)

cv_results = []

for name, model in models.items():
    scores = cross_val_score(model, X_full_tfidf, y, cv=5)
    cv_results.append({
        'Model': name,
        'CV Mean Accuracy': round(scores.mean(), 4),
        'CV Std Dev': round(scores.std(), 4),
        'CV Scores': [round(s, 4) for s in scores]
    })
    print(f'{name:25} CV Mean: {scores.mean():.2%}  ± {scores.std():.2%}  Scores: {[round(s, 4) for s in scores]}')

cv_results_df = pd.DataFrame(cv_results)
cv_results_df

## 8. Visual Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

metrics_to_plot = ['Accuracy', 'F1-score (Fake)', 'F1-score (Factual)']
x = np.arange(len(results_df['Model']))
width = 0.25

for i, metric in enumerate(metrics_to_plot):
    bars = ax.bar(x + i * width, results_df[metric], width, label=metric)

ax.set_xticks(x + width)
ax.set_xticklabels(results_df['Model'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Test Set Metrics')
ax.legend(loc='lower right')

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

cv_means = [r['CV Mean Accuracy'] for r in cv_results]
cv_stds = [r['CV Std Dev'] for r in cv_results]
model_names = [r['Model'] for r in cv_results]

bars = ax.bar(model_names, cv_means, yerr=cv_stds, capsize=8, color='steelblue')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Mean CV Accuracy')
ax.set_title('5-Fold Cross-Validation Comparison')

for bar, mean in zip(bars, cv_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{mean:.2%}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()

## 9. Select the Best Model

We'll choose the model with the best balance of test-set F1-score and CV accuracy,
then move it to Phase 4 for hyperparameter tuning.

In [ ]:
# Combine test metrics and CV metrics into a single ranking
comparison = results_df.merge(cv_results_df[['Model', 'CV Mean Accuracy']], on='Model')
comparison['Avg Score'] = comparison[['Accuracy', 'F1-score (Fake)', 'F1-score (Factual)', 'CV Mean Accuracy']].mean(axis=1)
comparison = comparison.sort_values('Avg Score', ascending=False).reset_index(drop=True)

comparison

best_model_name = comparison.iloc[0]['Model']
print(f'Best overall model: {best_model_name}')